# 17 (DE) — Operations & Observability

**Data Engineer perspective.** Running pipelines you can trust: caching hot DataFrames, reading execution plans, tracing lineage, per-query metrics, Python UDF registration, and the health-check CLI.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Working set

In [ ]:
import random
import pandas as pd

random.seed(3)
vendas_big = session.createDataFrame(pd.DataFrame({
    "evento_id": range(1, 2001),
    "estado": [random.choice(["SP", "RJ", "MG"]) for _ in range(2000)],
    "valor": [round(random.uniform(10, 500), 2) for _ in range(2000)],
}))
base = vendas_big.filter("valor > 50")

## 2. Cache hot results

`cache()` materializes a filtered result once; repeated actions stop re-scanning. `unpersist` releases it.

In [ ]:
cached = base.cache()
print("run 1:", cached.groupBy("estado").count().orderBy("estado").collect())
print("run 2 (cached):", cached.groupBy("estado").count().orderBy("estado").collect())
cached.unpersist()
print("unpersisted")

## 3. Execution plans

`explain()` prints the generated SQL plus IRIS's plan; `extended=True` adds the logical plan and execution-engine mapping.

In [ ]:
base.groupBy("estado").agg({"valor": "sum"}).explain()

In [ ]:
base.groupBy("estado").agg({"valor": "sum"}).explain(extended=True)

## 4. Lineage — the transformation recipe

Every lazy step is recorded; `lineage` replays it for debugging.

In [ ]:
vendas_big.filter("valor > 100").groupBy("estado").count().lineage(show=True)

## 5. Per-query metrics

Opt-in observability records timing/row metrics per executed statement.

In [ ]:
session.config("irispark.observability", True)
base.groupBy("estado").count().to_pandas()
metrics = getattr(session, "_metrics", [])
print("metrics recorded:", len(metrics))
for m in metrics[-2:]:
    print(m)
session.config("irispark.observability", False)

## 6. Registering a Python UDF

Python callables become SQL-callable functions via `session.udf.register`. Built-in ObjectScript/EPython UDF packs install through `irispark.sql.udf`.

In [ ]:
try:
    session.udf.register("double_it", lambda x: x * 2)
    print("registered double_it ->", session.udf._get("double_it")(21))
except Exception as e:
    print("udf registration note:", e)

## 7. Health check — `irispark-doctor`

From a terminal:

```bash
irispark-doctor
```

It checks connectivity, IRIS version, CPU feature flags, and columnar support.

**Performance rules of thumb** (`docs/performance_guide.md`): keep filters pushable (plain column comparisons), avoid correlated scalar subqueries per group, prefer single-pass analytics (`median`, window functions) over nested loops, and use columnar storage for wide analytical scans.

## 8. Wrap-up

You now have the full DE loop: ingest (14), store (15), audit (16), operate (17).

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")